##### SimpleAgent Demo

In [1]:
# test_simple_agent.py
from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, ToolRegistry, SimpleAgent
from hello_agents.tools import CalculatorTool

# 加载环境变量
load_dotenv()

# 创建LLM实例
llm = HelloAgentsLLM()

# 测试1:基础对话Agent（无工具）
print("=== 测试1:基础对话 ===")
basic_agent = SimpleAgent(
    name="基础助手",
    llm=llm,
    system_prompt="你是一个友好的AI助手，请用简洁明了的方式回答问题。"
)

response1 = basic_agent.run("你好，请介绍一下自己")
print(f"基础对话响应: {response1}\n")

# 测试2:带工具的Agent
print("=== 测试2:工具增强对话 ===")
tool_registry = ToolRegistry()
calculator = CalculatorTool()
tool_registry.register_tool(calculator)

enhanced_agent = SimpleAgent(
    name="增强助手",
    llm=llm,
    system_prompt="你是一个智能助手，可以使用工具来帮助用户。",
    tool_registry=tool_registry,
    enable_tool_calling=True
)

response2 = enhanced_agent.run("请帮我计算 15 * 8 + 32")
print(f"工具增强响应: {response2}\n")

# 测试3:流式响应
print("=== 测试3:流式响应 ===")
print("流式响应: ", end="")
# ✅ 用 for 循环遍历生成器，逐块输出
for chunk in basic_agent.run("请简单解释什么是人工智能", stream=True):
    print(chunk, end="", flush=True)
print()  # 最后换行



# 测试4:动态添加工具
print("\n=== 测试4:动态工具管理 ===")
print(f"添加工具前: {basic_agent.has_tools()}")
basic_agent.add_tool(calculator)
print(f"添加工具后: {basic_agent.has_tools()}")
print(f"可用工具: {basic_agent.list_tools()}")

# 查看对话历史
print(f"\n对话历史: {len(basic_agent.get_history())} 条消息")


=== 测试1:基础对话 ===
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
基础对话响应: 你好！我是一个友好的AI助手，由深度求索公司创造，专门用来回答你的问题、提供信息或协助完成任务。我乐于助人、简洁明了，可以陪你聊天、解答疑惑、出主意，甚至帮你写东西。有什么需要帮忙的吗？请随时告诉我！

=== 测试2:工具增强对话 ===
✅ 工具 'python_calculator' 已注册。
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
🧮 正在计算: 15*8+32
✅ 计算结果: 152
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
工具增强响应: 计算结果为 152。

=== 测试3:流式响应 ===
流式响应: 🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (流式)...
人工智能（AI）就是让计算机模拟人类智能的技术，比如学习、推理、解决问题。简单来说，它让机器能像人一样“思考”和“行动”，例如识别图片、对话交流或推荐视频。


=== 测试4:动态工具管理 ===
添加工具前: False
✅ 工具 'python_calculator' 已注册。
添加工具后: True
可用工具: ['python_calculator']

对话历史: 4 条消息


##### ReAct Demo

In [2]:
# test_react_agent.py
from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, ToolRegistry
from hello_agents.tools import CalculatorTool  
from hello_agents.agents import ReActAgent     

# 加载环境变量
load_dotenv()

# 1. 初始化LLM (推荐使用较强的模型)
llm = HelloAgentsLLM()

# 2. 初始化工具注册表并添加工具
registry = ToolRegistry()
registry.register_tool(CalculatorTool())

# 3. 初始化 ReActAgent
react_agent = ReActAgent(
    name="MathSolver",
    llm=llm,
    tool_registry=registry,
    max_steps=5  # 最多允许5轮 思考-行动 循环
)

# ============================
# 测试1: 基础多步推理与工具调用 (非流式)
# ============================
print("=" * 50)
print("🧪 测试1: ReAct 多步推理 (非流式)")
print("=" * 50)

question1 = "如果我一开始有15个苹果，别人又给了我8倍这么多，然后我又吃了32个，我还剩多少个苹果？"
# 正确逻辑：15 + 15 * 8 - 32 = 103
answer1 = react_agent.run(question1)
print(f"\n🎯 最终获取到的返回值: {answer1}\n")


# ============================
# 测试2: 需要多次调用工具的复杂计算 (非流式)
# ============================
print("=" * 50)
print("🧪 测试2: ReAct 连续工具调用 (非流式)")
print("=" * 50)

question2 = "请帮我算一下 123 加 456 的结果，然后再把这个结果乘以 2，然后再开根号向下取整"
answer2 = react_agent.run(question2)
print(f"\n🎯 最终获取到的返回值: {answer2}\n")


# ============================
# 测试3: 流式输出最终答案
# ============================
print("=" * 50)
print("🧪 测试3: ReAct 流式输出")
print("=" * 50)

question3 = "请帮我算一下 324 减 172 的结果，然后再把这个结果乘以 2，然后再开根号向上取整"
print("流式输出的最终答案: ", end="", flush=True)

# stream=True 时，中间的 Thought/Action/Observation 仍然会实时 print 出来
# 只有最终的 Finish 结果会以生成器返回
for chunk in react_agent.run(question3, stream=True):
    print(chunk, end="", flush=True)
print("\n")


# ============================
# 测试4: 不需要工具的普通问答
# ============================
print("=" * 50)
print("🧪 测试4: 无需工具的推理")
print("=" * 50)

question4 = "中国的首都是哪个城市？"
answer4 = react_agent.run(question4)
print(f"\n🎯 最终获取到的返回值: {answer4}\n")


✅ 工具 'python_calculator' 已注册。
🧪 测试1: ReAct 多步推理 (非流式)

🤖 MathSolver 开始处理问题: 如果我一开始有15个苹果，别人又给了我8倍这么多，然后我又吃了32个，我还剩多少个苹果？

--- 第 1 步 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
🤔 思考: 问题要求计算初始15个苹果，加上别人给的8倍（即15×8=120个），再减去吃掉的32个，最终剩余数量。这是一个简单的算术问题，可以直接计算。
🎬 行动: python_calculator[15 + 15*8 - 32]
🧮 正在计算: 15 + 15*8 - 32
✅ 计算结果: 103
👀 观察: 103

--- 第 2 步 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
🤔 思考: 根据计算，初始15个苹果，得到8倍即120个，总共135个，吃掉32个，剩余103个。结果已确认。
🎉 最终答案: 你还剩103个苹果。

🎯 最终获取到的返回值: 你还剩103个苹果。

🧪 测试2: ReAct 连续工具调用 (非流式)

🤖 MathSolver 开始处理问题: 请帮我算一下 123 加 456 的结果，然后再把这个结果乘以 2，然后再开根号向下取整

--- 第 1 步 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
🤔 思考: 需要计算 (123+456)*2 的平方根，然后向下取整。可以直接用工具计算整个表达式。
🎬 行动: python_calculator[floor(sqrt(2*(123+456)))]
🧮 正在计算: floor(sqrt(2*(123+456)))
❌ 计算失败: 不支持的函数: floor
👀 观察: 计算失败: 不支持的函数: floor

--- 第 2 步 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
🤔 思考: 之前尝试使用floor函数失败，因为工具不支持。现在我们可以先计算sqrt(2*(123+456))，得到数值后手动向下取整。
🎬 行动: python_c

##### Plan_and_Solve Agent Demo

In [1]:
from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, ToolRegistry
from hello_agents.tools import CalculatorTool, SearchTool
from hello_agents.agents import PlanAndSolveAgent

load_dotenv()

# 1. 初始化组件
llm = HelloAgentsLLM() # 建议使用 7B/8B 以上模型
registry = ToolRegistry()
registry.register_tool(CalculatorTool())
registry.register_tool(SearchTool())

# 2. 初始化 Agent
plan_agent = PlanAndSolveAgent(
    name="商业分析专家",
    llm=llm,
    tool_registry=registry
)

# 3. 执行复杂测试
print("=" * 60)
print("🧪 测试: 跨领域综合分析 (RAG检索 + 数学推导)")
print("=" * 60)

complex_question = """
请帮我完成一份简短的商业分析报告，具体要求如下：
1. 分别查询"长飞光纤"和"中芯国际"这两家公司2023年的财报数据。
2. 计算这两家公司2023年的总营收之和。
3. 假设行业平均总营收为 100 亿元，计算这两家公司的总营收之和分别比行业平均值高出百分之几。
"""

# 预期规划步骤：
# 步骤1: 查询星辰科技财报 -> 调用 rag_search
# 步骤2: 查询云端互联财报 -> 调用 rag_search
# 步骤3: 计算总营收之和 (85 + 62) -> 调用 python_calculator
# 步骤4: 计算超出行业平均的百分比 ((147 - 100) / 100 * 100) -> 调用 python_calculator

answer = plan_agent.run(complex_question)

print("\n" + "=" * 60)
print(f"📋 最终分析结论:\n{answer}")
print("=" * 60)


✅ 工具 'python_calculator' 已注册。
✅ Tavily 搜索引擎已初始化
✅ SerpApi 搜索引擎已初始化
🔧 混合搜索模式已启用，可用后端: tavily, serpapi
✅ 工具 'search' 已注册。
🧪 测试: 跨领域综合分析 (RAG检索 + 数学推导)

🤖 商业分析专家 开始处理问题: 
请帮我完成一份简短的商业分析报告，具体要求如下：
1. 分别查询"长飞光纤"和"中芯国际"这两家公司2023年的财报数据。
2. 计算这两家公司2023年的总营收之和。
3. 假设行业平均总营收为 100 亿元，计算这两家公司的总营收之和分别比行业平均值高出百分之几。


--- 正在生成计划 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
✅ 计划已生成:
```python
["使用search工具查询长飞光纤2023年的财报数据，获取其总营收", "使用search工具查询中芯国际2023年的财报数据，获取其总营收", "使用python_calculator计算两家公司2023年总营收之和", "使用python_calculator计算两家公司总营收之和比行业平均值（100亿元）高出百分之几"]
```

--- 正在执行计划 ---

-> 正在执行步骤 1/4: 使用search工具查询长飞光纤2023年的财报数据，获取其总营收
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
  🔧 检测到工具调用: search(长飞光纤 2023年 财报 总营收)
  👀 工具返回: 🔍 搜索关键词：长飞光纤 2023年 财报 总营收
🧭 使用搜索源：tavily

📚 参考来源：
[1] 长飞光纤Q1净利环比增长43% 去年光模块等组件贡献约两成营收|财报解读
    ## 首页## 电报## 话题## 盯盘## VIP## FM## 投研## 下载. 2026-04-29 22:20 星期三. > 面对每天上千份上市公司公告该看哪些？重大事项公告动辄几十页几百页重点是啥？公告里一堆专业术语不知道算利好还是利空？请看财联社公司新闻部《速读公告》栏目，我们派驻全国的记者们将于公告当晚为您带来准确、快速、专业的解读。. **财联社

##### Reflection Agent Demo

In [1]:
from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, ToolRegistry
from hello_agents.tools import CalculatorTool, SearchTool
from hello_agents.agents import ReflectionAgent

load_dotenv()

# 1. 初始化组件
llm = HelloAgentsLLM() # 建议使用代码能力较强的模型
registry = ToolRegistry()
registry.register_tool(CalculatorTool())
registry.register_tool(SearchTool())

# 2. 初始化 Reflection Agent
reflect_agent = ReflectionAgent(
    name="资深财务开发",
    llm=llm,
    tool_registry=registry,
    max_iterations=3
)

# 3. 执行复杂测试
print("=" * 60)
print("🧪 测试: 代码编写 -> 怀疑参数过期 -> 联网搜索 -> 修正代码 -> 验证")
print("=" * 60)

complex_task = """
请帮我完成以下复杂的编程与架构评估任务：
1. 编写一个 Python 函数 `fetch_all_users()`，使用 requests 库调用 NovaDB API 获取所有用户数据。
2. 该 API 有分页机制和速率限制，代码必须健壮地处理这些逻辑（包括遇到 429 错误时的重试等待）。
4. 只给出最终的完整 Python 代码。
"""

answer = reflect_agent.run(complex_task, stream=True)


✅ 工具 'python_calculator' 已注册。
✅ Tavily 搜索引擎已初始化
✅ SerpApi 搜索引擎已初始化
🔧 混合搜索模式已启用，可用后端: tavily, serpapi
✅ 工具 'search' 已注册。
🧪 测试: 代码编写 -> 怀疑参数过期 -> 联网搜索 -> 修正代码 -> 验证

🤖 资深财务开发 开始处理任务: 
请帮我完成以下复杂的编程与架构评估任务：
1. 编写一个 Python 函数 `fetch_all_users()`，使用 requests 库调用 NovaDB API 获取所有用户数据。
2. 该 API 有分页机制和速率限制，代码必须健壮地处理这些逻辑（包括遇到 429 错误时的重试等待）。
4. 只给出最终的完整 Python 代码。


--- 正在进行初始尝试 ---
🧠 正在调用 deepseek-ai/DeepSeek-V4-Flash 模型 (非流式)...
```python
import requests
import time
from typing import List, Dict, Any

def fetch_all_users(base_url: str = "https://api.novadb.com/users",
                    per_page: int = 100,
                    max_retries: int = 5,
                    timeout: int = 30) -> List[Dict[str, Any]]:
    """
    使用 requests 库调用 NovaDB API 获取所有用户数据。
    处理分页和速率限制（遇到 429 错误时重试等待）。

    Args:
        base_url: API 端点 URL
        per_page: 每页用户数量（根据 API 默认设置）
        max_retries: 最大重试次数
        timeout: 请求超时时间（秒）

    Returns:
        所有用户的列表
    """
    all_users = []
    page = 1
    r

INFO:openai._base_client:Retrying request to /chat/completions in 0.421686 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 0.758297 seconds


当前回答基本满足要求，但存在若干可改进之处：

1. **Retry-After 头处理不完整**  
   - 仅检查 `isdigit()`，但 `Retry-After` 可能是 HTTP-date 格式（如 `"Wed, 21 Oct 2015 07:28:00 GMT"`），此时 `isdigit()` 返回 `False`，会退回到指数退避，可能忽略服务器建议的等待时间。  
   - 建议：使用 `try: int(retry_after)` 尝试解析，若失败则使用指数退避，或引入 `email.utils.parsedate` 解析日期格式。

2. **错误类型区分不足**  
   - 对非 429 的 HTTP 错误（如 400、403、500）也进行了重试，这可能导致不必要的等待或无限重试。例如客户端错误（4xx）通常不应重试。  
   - 建议：在 `except` 块中区分 `requests.exceptions.HTTPError`，对 4xx（除 429）和 5xx 分别处理，或直接抛出异常。

3. **重试计数器共享问题**  
   - 同一 `retries` 变量同时用于 429 重试和网络异常重试，可能导致一种错误消耗了重试次数后，另一种错误过早放弃。  
   - 建议：分别维护两个计数器（`rate_limit_retries` 和 `network_retries`），或重置网络异常重试计数器（类似成功后的重置）。

4. **无限循环风险**  
   - 当 API 不返回 `total_pages` 或 `total`，且每页用户数恰好等于 `per_page`（包括最后一页）时，循环无法退出，会一直请求下一页直到超时或重试耗尽。  
   - 建议：增加最大页数限制（如 `max_pages = 1000`）作为安全阀，或检测 `next` 链接字段。

5. **响应格式假设过于刚性**  
   - 假设响应永远是 `{"data": [...], ...}`，但实际 API 可能直接返回列表或使用其他键名（如 `"users"`）。  
   - 建议：检查 `response.json()` 类型，若是列表直接赋值；若是字典，可接受一个 `data_key` 参数（如 `"data"`）或自动尝试

INFO:openai._base_client:Retrying request to /chat/completions in 0.424402 seconds


好的，根据您的反馈意见，我对代码进行了改进。主要改进点包括：更健壮的 `Retry-After` 头解析、区分错误类型并采用不同的重试策略、使用独立的计数器、增加最大页数安全阀、更灵活的响应格式处理、使用 `logging` 模块以及更具体的异常处理。

以下是改进后的完整代码：

```python
import requests
import time
import logging
from typing import List, Dict, Any, Optional
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone

# 配置日志记录器
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def fetch_all_users(base_url: str = "https://api.novadb.com/users",
                    per_page: int = 100,
                    max_retries: int = 5,
                    timeout: int = 30,
                    data_key: str = "data") -> List[Dict[str, Any]]:
    """
    使用 requests 库调用 NovaDB API 获取所有用户数据。
    健壮地处理分页、速率限制（429错误）和其他网络/HTTP错误。

    Args:
        base_url: API 端点 URL。
        per_page: 每页用户数量。
        max_retries: 最大重试次数（分别应用于速率限制、网络超时和连接错误）。
        timeout: 请求超时时间（秒）。
        data_key: 响应字典中包含用户列表的键名。如果响应直接是列表，

INFO:openai._base_client:Retrying request to /chat/completions in 0.471856 seconds


这个回答整体质量很高，代码结构清晰、功能完善、异常处理全面。但仍存在一些可以改进的地方：

## 主要问题与改进建议

### 1. **`network_retries` 未在成功页面后重置**
虽然代码中重置了 `rate_limit_retries`、`json_parse_retries`、`server_error_retries`，但**遗漏了 `network_retries` 的重置**。这会导致网络错误计数器在成功获取一页后不会归零，后续网络错误会继承之前的计数，可能导致过早放弃重试。

**改进建议**：在成功处理一页后的重置部分添加：
```python
network_retries = 0
```

### 2. **网络重试与分页逻辑的交互问题**
当网络错误发生在获取第N页时，`page` 变量没有递增，但重试成功后会继续处理同一页。这本身是正确的。但问题在于：
- 如果网络错误发生在**分页终止判断之后**（即 `page += 1` 之前），重试时会重新进入while循环，但此时 `page` 值未变，会重复处理同一页。
- 这实际上不是bug，但逻辑上可以更清晰：将网络重试放在try块外部，或使用更明确的状态机。

### 3. **`total_pages` 和 `total` 的获取逻辑冗余**
代码中两次判断 `isinstance(body, dict)` 来获取 `total_pages` 和 `total`，可以合并为一次判断：

```python
if isinstance(body, dict):
    total_pages = body.get("total_pages")
    total = body.get("total")
```

### 4. **`max_pages` 安全阀的警告不够准确**
当达到 `max_pages` 时，警告信息中的 `total_pages_fetched` 实际上等于 `max_pages`，但用户可能期望知道实际获取了多少页数据。更准确的表述应该是：
```python
logger.warning(f"已达到最大页数限制 ({max_pages})，可能未获取全部数据。实际获取了 {total_pages_fetched} 页。")
```

### 5. 

INFO:openai._base_client:Retrying request to /chat/completions in 0.420209 seconds


以下是根据反馈意见改进后的完整代码，主要修正了 `network_retries` 未重置、分页逻辑冗余、`Retry-After` 为 0 时无最小等待、`per_page` 参数验证、`max_pages` 警告信息不准确、参数命名一致性、类型注解注释增强等问题：

```python
import requests
import time
import logging
from typing import List, Dict, Any, Optional
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone
import json

# 配置模块级日志器，不设置全局处理器，避免干扰外部调用者
logger = logging.getLogger(__name__)
logger.addHandler(logging.NullHandler())

def fetch_all_users(base_url: str = "https://api.novadb.com/users",
                    per_page: int = 100,
                    max_retries: int = 5,
                    timeout: int = 30,
                    data_key: str = "data",
                    retry_on_5xx: bool = True,
                    max_wait_seconds: int = 300) -> List[Dict[str, Any]]:
    """
    使用 requests 库调用 NovaDB API 获取所有用户数据。
    健壮地处理分页、速率限制（429 错误）、临时性 5xx 错误、超时、连接错误及 JSON 解析错误。

    Args:
        base_url: API 端点 URL。
        per_page: 每页用户数量（必须为正整数）。
        max_retries: 最大